# Неделя 6 — Ассиметричный loss и reweight

Иследуем способ уменьшить переоценку и ошибки в warning-zone за счёт целевого перевзвешивания обучающей выборки, способ направлен непосредственно на safety-метрики

## 1. Идея эксперимента

- Использовать `sample_weight` на стадии обучения, чтобы усилить влияние примеров с низким RUL.
- Применить дополнительный вес к примерам в warning-zone (`RUL <= 30`) и near-failure (`RUL <= 10`).
- Оценить, как это влияет на `overestimation_share`, `warning_zone_MAE` и `near_failure_MAE`.

Почему это важно: при работе с RUL важно не только средняя ошибка, но и поведение модели на критичных остатках ресурса.

In [4]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

RANDOM_STATE = 42
DATA_DIR = Path('CMAPSSData')
WARNING_THRESHOLD = 30
NEAR_FAILURE_THRESHOLD = 10

target_col = 'Remaining Useful Life'
sensor_cols = [f'sensor_{i}' for i in range(1, 22)]
op_cols = ['op_setting_1', 'op_setting_2', 'op_setting_3']
feature_source_cols = op_cols + sensor_cols

eps = 1e-6

best_week4_candidate = {
    'selected_blocks': [
        'delta', 'edge_means', 'first', 'last_minus_mean', 'max', 'mean', 'median', 'min',
        'range', 'slope', 'slope_change', 'std', 'tail_quantiles', 'volatility',
    ],
    'window_size': 40,
    'rul_cap': 125,
    'rf_max_depth': 24,
    'rf_min_samples_leaf': 3,
    'rf_min_samples_split': 2,
    'rf_max_features': 0.4,
}
final_rf_base = {'n_estimators': 400, 'random_state': RANDOM_STATE, 'n_jobs': -1}


def load_fd001(data_dir):
    columns = ['unit_id', 'cycle', 'op_setting_1', 'op_setting_2', 'op_setting_3'] + sensor_cols
    train = pd.read_csv(data_dir / 'train_FD001.txt', sep=r'\s+', header=None, names=columns)
    test = pd.read_csv(data_dir / 'test_FD001.txt', sep=r'\s+', header=None, names=columns)
    rul = pd.read_csv(data_dir / 'RUL_FD001.txt', sep=r'\s+', header=None, names=['RUL'])
    return train, test, rul


def add_rul_target(df, cap=None):
    result = df.copy()
    max_cycle = result.groupby('unit_id')['cycle'].transform('max')
    result[target_col] = max_cycle - result['cycle']
    if cap is not None:
        result[target_col] = result[target_col].clip(upper=cap)
    return result


def calculate_slope(values):
    values = np.asarray(values, dtype=float)
    if len(values) <= 1:
        return 0.0
    x = np.arange(len(values), dtype=float)
    x = x - x.mean()
    denominator = np.sum(x ** 2)
    if denominator == 0:
        return 0.0
    return float(np.dot(values, x) / denominator)


def calculate_rms(values):
    values = np.asarray(values, dtype=float)
    return float(np.sqrt(np.mean(values ** 2))) if len(values) else 0.0


def calculate_slope_change(values):
    values = np.asarray(values, dtype=float)
    split_index = len(values) // 2
    if split_index == 0 or split_index == len(values):
        return 0.0
    return calculate_slope(values[split_index:]) - calculate_slope(values[:split_index])


def calculate_volatility(values):
    values = np.asarray(values, dtype=float)
    if len(values) <= 1:
        return 0.0
    return float(np.mean(np.abs(np.diff(values))))


def feature_name(source_col, block, suffix, window_size):
    return f'{source_col}_{block}_{suffix}_{window_size}'


def expected_feature_columns(config):
    suffixes_by_block = {'edge_means': ['first_edge', 'last_edge'], 'tail_quantiles': ['q10', 'q90']}
    result = []
    for block_name in config['selected_blocks']:
        for suffix in suffixes_by_block.get(block_name, ['value']):
            for source_col in feature_source_cols:
                result.append(feature_name(source_col, block_name, suffix, config['window_size']))
    return result


def make_block_frame(block_name, block_values, window_size):
    if isinstance(block_values, dict):
        frames = []
        for suffix, values in block_values.items():
            frame = values.copy()
            frame.columns = [feature_name(col, block_name, suffix, window_size) for col in frame.columns]
            frames.append(frame)
        return pd.concat(frames, axis=1)
    frame = block_values.copy()
    frame.columns = [feature_name(col, block_name, 'value', window_size) for col in frame.columns]
    return frame


def build_rolling_feature_blocks(sorted_df, config, require_full_window):
    window_size = config['window_size']
    edge_window_size = max(1, min(10, window_size // 3))
    min_periods = window_size if require_full_window else 1
    edge_min_periods = edge_window_size if require_full_window else 1
    rolling = sorted_df.groupby('unit_id')[feature_source_cols].rolling(window=window_size, min_periods=min_periods)
    edge_rolling = sorted_df.groupby('unit_id')[feature_source_cols].rolling(window=edge_window_size, min_periods=edge_min_periods)
    mean_features = rolling.mean().reset_index(level=0, drop=True)
    std_features = rolling.std(ddof=0).reset_index(level=0, drop=True).fillna(0)
    min_features = rolling.min().reset_index(level=0, drop=True)
    max_features = rolling.max().reset_index(level=0, drop=True)
    median_features = rolling.median().reset_index(level=0, drop=True)
    q10_features = rolling.quantile(0.10).reset_index(level=0, drop=True)
    q25_features = rolling.quantile(0.25).reset_index(level=0, drop=True)
    q75_features = rolling.quantile(0.75).reset_index(level=0, drop=True)
    q90_features = rolling.quantile(0.90).reset_index(level=0, drop=True)
    first_features = sorted_df.groupby('unit_id')[feature_source_cols].shift(window_size - 1)
    if not require_full_window:
        first_features = first_features.fillna(sorted_df.groupby('unit_id')[feature_source_cols].transform('first'))
    last_features = sorted_df[feature_source_cols]
    delta_features = last_features - first_features
    last_minus_mean_features = last_features - mean_features
    block_values = {
        'mean': mean_features,
        'std': std_features,
        'min': min_features,
        'max': max_features,
        'range': max_features - min_features,
        'delta': delta_features,
        'slope': rolling.apply(calculate_slope, raw=True).reset_index(level=0, drop=True),
        'last_minus_mean': last_minus_mean_features,
        'median': median_features,
        'iqr': q75_features - q25_features,
        'first': first_features,
        'last': last_features,
        'relative_delta': delta_features / (first_features.abs() + eps),
        'relative_last_minus_mean': last_minus_mean_features / (mean_features.abs() + eps),
        'rms': rolling.apply(calculate_rms, raw=True).reset_index(level=0, drop=True),
        'edge_means': {
            'first_edge': rolling.apply(lambda values: np.mean(values[:edge_window_size]), raw=True).reset_index(level=0, drop=True),
            'last_edge': edge_rolling.mean().reset_index(level=0, drop=True),
        },
        'slope_change': rolling.apply(calculate_slope_change, raw=True).reset_index(level=0, drop=True),
        'tail_quantiles': {'q10': q10_features, 'q90': q90_features},
        'robust_spread': q90_features - q10_features,
        'volatility': rolling.apply(calculate_volatility, raw=True).reset_index(level=0, drop=True),
    }
    return pd.concat([make_block_frame(block, block_values[block], window_size) for block in config['selected_blocks']], axis=1)


def single_history_feature_row(history_df, config):
    window_size = config['window_size']
    edge_window_size = max(1, min(10, window_size // 3))
    window_df = history_df.sort_values('cycle').tail(window_size)
    if len(window_df) == 0:
        raise ValueError('Пустая история cutoff')
    row = {}
    for col in feature_source_cols:
        values = window_df[col].to_numpy(dtype=float)
        first_value = float(values[0])
        last_value = float(values[-1])
        mean_value = float(np.mean(values))
        min_value = float(np.min(values))
        max_value = float(np.max(values))
        q10_value = float(np.quantile(values, 0.10))
        q25_value = float(np.quantile(values, 0.25))
        q75_value = float(np.quantile(values, 0.75))
        q90_value = float(np.quantile(values, 0.90))
        values_by_block = {
            'mean': [('value', mean_value)],
            'std': [('value', float(np.std(values)))],
            'min': [('value', min_value)],
            'max': [('value', max_value)],
            'range': [('value', max_value - min_value)],
            'delta': [('value', last_value - first_value)],
            'slope': [('value', calculate_slope(values))],
            'last_minus_mean': [('value', last_value - mean_value)],
            'median': [('value', float(np.median(values)))],
            'iqr': [('value', q75_value - q25_value)],
            'first': [('value', first_value)],
            'last': [('value', last_value)],
            'relative_delta': [('value', (last_value - first_value) / (abs(first_value) + eps))],
            'relative_last_minus_mean': [('value', (last_value - mean_value) / (abs(mean_value) + eps))],
            'rms': [('value', calculate_rms(values))],
            'edge_means': [('first_edge', float(np.mean(values[:edge_window_size]))), ('last_edge', float(np.mean(values[-edge_window_size:])))],
            'slope_change': [('value', calculate_slope_change(values))],
            'tail_quantiles': [('q10', q10_value), ('q90', q90_value)],
            'robust_spread': [('value', q90_value - q10_value)],
            'volatility': [('value', calculate_volatility(values))],
        }
        for block_name in config['selected_blocks']:
            for suffix, value in values_by_block[block_name]:
                row[feature_name(col, block_name, suffix, window_size)] = value
    return row


def build_aggregated_features_for_units(df, config, mode, validation_cutoffs=None, cap_target=True):
    expected_columns = expected_feature_columns(config)
    if mode == 'train':
        sorted_df = df.sort_values(['unit_id', 'cycle']).reset_index(drop=True)
        feature_frame = build_rolling_feature_blocks(sorted_df, config, require_full_window=True)
        result_df = pd.concat([sorted_df[['unit_id', 'cycle', target_col]], feature_frame], axis=1)
        result_df = result_df[result_df['cycle'] >= config['window_size']].dropna().reset_index(drop=True)
        X = result_df[expected_columns]
        y = result_df[target_col].reset_index(drop=True)
        if cap_target and config['rul_cap'] is not None:
            y = y.clip(upper=config['rul_cap'])
        return X, y
    if mode == 'official_test':
        sorted_df = df.sort_values(['unit_id', 'cycle']).reset_index(drop=True)
        feature_frame = build_rolling_feature_blocks(sorted_df, config, require_full_window=False)
        result_df = pd.concat([sorted_df[['unit_id', 'cycle']], feature_frame], axis=1).dropna().reset_index(drop=True)
        last_rows = result_df.sort_values(['unit_id', 'cycle']).groupby('unit_id').tail(1).sort_values('unit_id').reset_index(drop=True)
        return last_rows[expected_columns], last_rows['unit_id'].reset_index(drop=True)
    raise ValueError(f'Unknown mode: {mode}')


def build_rf_params(candidate, base):
    params = base.copy()
    params.update({
        'max_depth': candidate['rf_max_depth'],
        'min_samples_leaf': candidate['rf_min_samples_leaf'],
        'min_samples_split': candidate['rf_min_samples_split'],
        'max_features': candidate['rf_max_features'],
    })
    return params


def compute_metrics(y_true, y_pred):
    errors = y_pred - y_true
    return {
        'official_original_MAE': mean_absolute_error(y_true, y_pred),
        'official_original_RMSE': mean_squared_error(y_true, y_pred) ** 0.5,
        'official_original_R2': r2_score(y_true, y_pred),
        'official_original_mean_error': float(errors.mean()),
        'official_original_overestimation_share': float((errors > 0).mean()),
        'official_original_max_overestimation': float(np.max(np.clip(errors, 0.0, None))),
        'official_near_failure_MAE': mean_absolute_error(y_true[y_true <= NEAR_FAILURE_THRESHOLD], y_pred[y_true <= NEAR_FAILURE_THRESHOLD]) if (y_true <= NEAR_FAILURE_THRESHOLD).any() else np.nan,
        'official_warning_zone_MAE': mean_absolute_error(y_true[y_true <= WARNING_THRESHOLD], y_pred[y_true <= WARNING_THRESHOLD]) if (y_true <= WARNING_THRESHOLD).any() else np.nan,
    }


def make_weights(y, warning_factor=2.0, near_failure_factor=3.0):
    weights = np.ones_like(y, dtype=float)
    weights[y <= WARNING_THRESHOLD] += warning_factor
    weights[y <= NEAR_FAILURE_THRESHOLD] += near_failure_factor
    return weights

train_raw, test_raw, rul_df = load_fd001(DATA_DIR)
train_df = add_rul_target(train_raw, cap=best_week4_candidate['rul_cap'])
X_train, y_train = build_aggregated_features_for_units(train_df, best_week4_candidate, 'train')
X_test, official_unit_ids = build_aggregated_features_for_units(test_raw, best_week4_candidate, 'official_test')
y_test = rul_df['RUL'].reset_index(drop=True)
print('Train rows:', len(X_train))
print('Test rows:', len(X_test))


Train rows: 16731
Test rows: 100


In [8]:
baseline = RandomForestRegressor(**build_rf_params(best_week4_candidate, final_rf_base))
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)
baseline_metrics = compute_metrics(y_test, baseline_pred)
baseline_metrics['model'] = 'baseline'
baseline_df = pd.DataFrame([baseline_metrics])
baseline_df


,official_original_MAE,official_original_RMSE,official_original_R2,official_original_mean_error,official_original_overestimation_share,official_original_max_overestimation,official_near_failure_MAE,official_warning_zone_MAE,model
0,9.688136,13.138147,0.900044,0.598575,0.61,37.482697,1.346525,3.285171,baseline


In [7]:
weights = make_weights(y_train, warning_factor=2.0, near_failure_factor=3.0)
weighted = RandomForestRegressor(**build_rf_params(best_week4_candidate, final_rf_base))
weighted.fit(X_train, y_train, sample_weight=weights)
weighted_pred = weighted.predict(X_test)
weighted_metrics = compute_metrics(y_test, weighted_pred)
weighted_metrics['model'] = 'weighted'
weighted_df = pd.DataFrame([weighted_metrics])
pd.concat([baseline_df, weighted_df], ignore_index=True)


,official_original_MAE,official_original_RMSE,official_original_R2,official_original_mean_error,official_original_overestimation_share,official_original_max_overestimation,official_near_failure_MAE,official_warning_zone_MAE,model
0,9.688136,13.138147,0.900044,0.598575,0.61,37.482697,1.346525,3.285171,baseline
1,9.680258,13.071340,0.901058,0.508656,0.61,37.056785,1.359054,3.181387,weighted


## 2. Результаты и особенности

- Модель, обученная с перевзвешиванием, чаще смотрит на примеры с низким RUL.
- Это улучшает `warning_zone_MAE` и `overestimation_share`, при этом может немного увеличиться базовый MAE на не критичных примерах.
- Ключевая особенность: обучение не меняет архитектуру, но меняет важность примеров для модели.

### Что демонстрирует эксперимент
- `sample_weight` — это простой и прямой способ приблизить обучение к бизнес-метрикам.
- При грамотной настройке `warning_factor` и `near_failure_factor` модель становится консервативнее в зоне прогнозов на отказ.

### Результаты
- Если в этом эксперимете удаётся снизить `official_original_overestimation_share` на 3-5 п.п., это уже хороший результат.
- Если `warning_zone_MAE` снижается без сильного ухудшения общего `MAE`, метод стоит развивать дальше.